# Get the price data

The Dukascopy export form on their website downloads **one day at a time**. Two
years would be 730 downloads by hand.

The same data sits behind a plain HTTP feed, one file per hour, and this
notebook pulls the whole range in one go — on Google's machines, so nothing is
installed on your Chromebook.

**Runtime → Run all**, then read Step 3 before letting Step 4 run.

## Step 1 — Get the code

In [ ]:
import os, shutil, subprocess

REPO   = "https://github.com/dboy140/Dboytrades.git"
BRANCH = "claude/ict-nbbtrader-trading-system-43hipg"

if os.path.isdir("/content/Dboytrades"):
    shutil.rmtree("/content/Dboytrades")

r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO, "/content/Dboytrades"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit(f"Could not download the code:\n{r.stderr}")

os.chdir("/content/Dboytrades")
%pip install -q --upgrade pydantic
print("Step 1 done.")

## Step 2 — Choose what to download

`EURUSD` and `GBPUSD` are the pairs the NBB rules are stated for. `XAUUSD` is
covered by neither source (GAPS G-04), so treat anything it produces as
exploratory.

Two years is the target. That is not a round number picked for neatness — the
trade windows are about an hour a day and a setup does not appear every day, so
a shorter file produces a *confident wrong* answer rather than a cautious one.

In [ ]:
from datetime import datetime, timezone

SYMBOL = "EURUSD"        # EURUSD | GBPUSD | XAUUSD | USDJPY ...
START  = datetime(2024, 1, 1, tzinfo=timezone.utc)
END    = datetime(2026, 1, 1, tzinfo=timezone.utc)
SIDE   = "bid"           # match the offer side you'd trade

from bot.dukascopy import hours_between
n = len(list(hours_between(START, END)))
print(f"{SYMBOL} {SIDE}: {START.date()} -> {END.date()}")
print(f"{n:,} hourly files to fetch (weekend hours come back empty)")

## Step 3 — Check one hour first

Read this before running Step 4.

This fetches a **single** hour and prints what came back. If the feed has moved
or the format has changed, you find out in five seconds rather than forty
minutes.

What you should see: a few thousand ticks, and a price that looks like a real
quote for your instrument. **If the price looks wrong by a factor of 100 or
1000, stop** — that is the point-factor table in `bot/dukascopy.py` not knowing
your symbol, and every bar in the file would be scaled wrong while still
looking perfectly well-formed.

In [ ]:
from datetime import datetime, timezone
from bot.dukascopy import fetch_hour, decode_bi5, hour_url, ticks_to_minutes

probe = datetime(2024, 3, 5, 14, tzinfo=timezone.utc)   # a Tuesday, London pm
print("URL:", hour_url(SYMBOL, probe))

raw = fetch_hour(SYMBOL, probe)
print(f"bytes returned: {len(raw):,}")

if not raw:
    print("\nEMPTY. That hour has no data. Try a different weekday/hour,")
    print("or the symbol may not exist on this feed under that name.")
else:
    ticks = decode_bi5(raw, SYMBOL, probe)
    print(f"ticks decoded : {len(ticks):,}")
    t = ticks[0]
    print(f"\nfirst tick    : {t.ts}")
    print(f"  bid {t.bid}   ask {t.ask}   spread {t.ask - t.bid:.5f}")
    rows = ticks_to_minutes(ticks, SIDE)
    print(f"\n1-minute bars in this hour: {len(rows)}")
    print("first bar:", rows[0])
    print("\nDoes that bid look like a real price for", SYMBOL + "?")
    print("If yes, run Step 4. If it is out by 100x or 1000x, stop and say so.")

## Step 4 — Download the full range

Expect several minutes for two years. Progress prints as it goes.

Hours when the market is shut return empty and are skipped — that is cheaper
and more accurate than encoding a holiday calendar.

In [ ]:
import time
from bot.dukascopy import fetch_range

def show(done, total):
    print(f"  {done:,}/{total:,} hours ({done/total:.0%})")

t0 = time.time()
rows = fetch_range(SYMBOL, START, END, side=SIDE, workers=16, progress=show)
print(f"\n{len(rows):,} one-minute bars in {time.time() - t0:.0f}s")
if rows:
    print(f"from {rows[0][0]} to {rows[-1][0]}")

## Step 5 — Write the CSV

In [ ]:
import pathlib
from bot.dukascopy import rows_to_csv

pathlib.Path("data/bars").mkdir(parents=True, exist_ok=True)
name = f"{SYMBOL}_1m_{START.date()}_{END.date()}.csv"
CSV = f"data/bars/{name}"
pathlib.Path(CSV).write_text(rows_to_csv(rows))

mb = pathlib.Path(CSV).stat().st_size / 1e6
print(f"wrote {CSV}  ({mb:.1f} MB, {len(rows):,} bars)")

## Step 6 — Is the file any good?

Bars alone are not enough. This checks the timestamps, looks for duplicates and
gaps, and counts how many bars actually fall inside each trade window.

**The session coverage numbers are the ones that matter.** A file with a million
rows and zero bars in the Silver Bullet window cannot test `SB-001`, however
big it is.

The timezone check should report no drift here — this feed is already UTC and
the timestamps carry their offset. That check exists for broker exports, which
are stamped in server time and would put every session window hours out.

In [ ]:
!python -m bot.inspect_data "$CSV" 

## Step 7 — Download it to your Chromebook

In [ ]:
try:
    from google.colab import files
    files.download(CSV)
except Exception as exc:
    print(f"Auto-download unavailable ({exc}).")
    print("Use the folder icon in the left sidebar and download it from data/bars/.")

---

## Then what

Two options:

1. **Keep going here.** Open `notebooks/validate_colab.ipynb`, upload this CSV
   at its Step 3, and run the walk-forward validation.
2. **Send it to me.** Attach the CSV in the chat and I'll run it.

Either way, the thing to expect is a `FAILED` verdict. That is the honest base
rate for a mechanical rule set on out-of-sample data, and the validator is built
to say it — it currently refuses a pure random walk, which the previous version
did not. A pass would be the surprise.